# 10 · Baselines — Buy-Again & Popularity

**Purpose.** Establish the non-ML reference scores that every later model
(item-kNN, ALS, XGBoost ranker) must beat, and fix the evaluation protocol
they will all be measured under. A model that cannot clear a heuristic that
runs in one SQL query has no business in production.

## Evaluation protocol

For a snapshot day `as_of`: build top-K recommendations per household using
only information from `day_no ≤ as_of`, then compare against the products the
household actually bought in `(as_of, as_of + 30]` (via `purchase_labels`).
Baselines have no parameters to fit, so they are scored on the validation
snapshot (day 600); the test snapshot (650) stays sealed until final
comparison.

## Metrics

- **Recall@K** — per household: (recommended ∩ bought) / (all bought), then
  averaged. *Ceiling caveat:* households buy far more than K products in 30
  days, so even a perfect recommender scores well below 1.0 — we therefore
  also report the **oracle ceiling** E[min(K, n_bought)/n_bought] and judge
  models as a fraction of it.
- **HitRate@K** — share of households whose top-K contains ≥1 actual
  purchase; the "was the list useful at all" metric.
- **NDCG@K** — rank-weighted (hits near the top count more), normalized by
  the ideal ordering; binary relevance.

## Baselines

1. **Buy-again** — rank each household's own past products by recency, then
   frequency. Justified by EDA: repeat purchases are ~55% of mature volume,
   and the due-ness curve showed recency is the dominant signal at a 30-day
   horizon.
2. **Popularity** — global top-K and department-localized top-K. Justified as
   the cold-start fallback; EDA (flat head: top-10 products = 4.7% of lines)
   predicts global popularity will be weak here.

In [1]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from pathlib import Path

from retail_ds.evaluate.metrics import recall_at_k, hit_rate_at_k, ndcg_at_k

# --- repo root (works from notebooks/10_baselines/ or the repo root) ---
ROOT = Path.cwd()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parents[1]

# --- config: single source of truth for dates/horizons ---
CFG = yaml.safe_load((ROOT / "configs" / "base.yaml").read_text())

# --- read-only DB connection + one-line query helper ---
con = duckdb.connect((ROOT / "db" / "retail.duckdb").as_posix(), read_only=True)

def q(sql: str) -> pd.DataFrame:
    """Run SQL against retail.duckdb and return a pandas DataFrame."""
    return con.sql(sql).df()

# --- plotting defaults ---
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 4.5)

# --- sanity echo: fail loudly here, not three sections later ---
print("day range :", q("SELECT MIN(day_no) lo, MAX(day_no) hi FROM staging.stg_transactions").iloc[0].tolist())
print("snapshots :", CFG["snapshots"], "| horizon:", CFG["label_horizon_days"])

day range : [1, 711]
snapshots : {'train': [450, 510, 570], 'valid': 600, 'test': 650} | horizon: 30


In [2]:
AS_OF = CFG["snapshots"]["valid"]   # 600 — baselines need no training; test day 650 stays sealed
K = CFG["top_k"]

buy_again = q(f"""
    SELECT household_key, product_id,
           ROW_NUMBER() OVER (
               PARTITION BY household_key
               ORDER BY days_since_last ASC, times_bought DESC
           ) AS rank
    FROM household_product_snapshot({AS_OF})
""")
labels = q(f"SELECT household_key, product_id FROM purchase_labels({AS_OF}, 30)")

print(f"buy-again  recall@{K}: {recall_at_k(buy_again, labels, K):.3f}")
print(f"buy-again  hit_rate@{K}: {hit_rate_at_k(buy_again, labels, K):.3f}")
print(f"buy-again  ndcg@{K}: {ndcg_at_k(buy_again, labels, K):.3f}")

buy-again  recall@10: 0.072
buy-again  hit_rate@10: 0.757
buy-again  ndcg@10: 0.303


In [3]:
# oracle ceiling: perfect recommender limited to K slots
n_bought = labels.groupby("household_key").size()
oracle_recall = (n_bought.clip(upper=K) / n_bought).mean()
print(f"oracle recall@{K} ceiling: {oracle_recall:.3f}")

oracle recall@10 ceiling: 0.402


**Findings — buy-again baseline (as-of 600, K=10).**
recall@10 = 0.072 · hit_rate@10 = 0.759 · ndcg@10 = 0.303, against an oracle
ceiling of 0.402 — buy-again alone captures ≈18% of achievable recall with
zero machine learning and one SQL query; 76% of households get at least one
correct item in their top 10.

**Interpretation.** The absolute number is deflated by the many-items-per-month
denominator (a typical household buys ~25 distinct products/month, so the
ceiling is 0.402, not 1.0). Judged against the ceiling: a strong floor,
consistent with the EDA (55% repeat volume, recency dominance) — but with 82%
of achievable recall unclaimed. Buy-again is structurally capped: it can only
recommend what a household already bought. The discovery gap is the candidate
generators' job; ranking quality within the list is the ranker's.

In [4]:
global_pop = q(f"""
    WITH top_products AS (
        SELECT product_id, COUNT(*) AS n
        FROM staging.stg_transactions
        WHERE day_no <= {AS_OF} AND day_no > {AS_OF} - 365
        GROUP BY product_id
        ORDER BY n DESC LIMIT {K}
    ),
    households AS (
        SELECT DISTINCT household_key FROM customer_snapshot({AS_OF})
    )
    SELECT h.household_key, t.product_id,
           ROW_NUMBER() OVER (PARTITION BY h.household_key ORDER BY t.n DESC) AS rank
    FROM households h CROSS JOIN top_products t
""")

In [5]:
dept_pop = q(f"""
    WITH hh_dept AS (
        SELECT t.household_key, p.department, COUNT(*) AS dept_lines
        FROM staging.stg_transactions t
        JOIN staging.stg_products p USING (product_id)
        WHERE t.day_no <= {AS_OF} AND t.day_no > {AS_OF} - 365
        GROUP BY t.household_key, p.department
    ),
    dept_top AS (
        SELECT p.department, t.product_id, COUNT(*) AS n
        FROM staging.stg_transactions t
        JOIN staging.stg_products p USING (product_id)
        WHERE t.day_no <= {AS_OF} AND t.day_no > {AS_OF} - 365
        GROUP BY p.department, t.product_id
        QUALIFY ROW_NUMBER() OVER (PARTITION BY p.department ORDER BY COUNT(*) DESC) <= 20
    )
    SELECT household_key, product_id, rank FROM (
        SELECT h.household_key, d.product_id,
               ROW_NUMBER() OVER (
                   PARTITION BY h.household_key
                   ORDER BY h.dept_lines * d.n DESC
               ) AS rank
        FROM hh_dept h JOIN dept_top d USING (department)
    )
    WHERE rank <= {K}
""")

In [6]:
results = {"buy_again": buy_again, "global_popularity": global_pop, "dept_popularity": dept_pop}

rows = []
for name, recs in results.items():
    rows.append({"model": name,
                 f"recall@{K}":   recall_at_k(recs, labels, K),
                 f"hit_rate@{K}": hit_rate_at_k(recs, labels, K),
                 f"ndcg@{K}":     ndcg_at_k(recs, labels, K)})

leaderboard = pd.DataFrame(rows).set_index("model").round(3)
leaderboard["share_of_ceiling"] = (leaderboard[f"recall@{K}"] / oracle_recall).round(2)
leaderboard.to_csv(ROOT / "reports" / "baseline_leaderboard.csv")
leaderboard

,recall@10,hit_rate@10,ndcg@10,share_of_ceiling
model,,,,
buy_again,0.072,0.757,0.303,0.18
global_popularity,0.041,0.703,0.192,0.10
dept_popularity,0.034,0.650,0.155,0.08


**Findings — baseline leaderboard (as-of 600, K=10, oracle ceiling 0.402).**

| model | recall@10 | hit_rate@10 | ndcg@10 | share of ceiling |
|---|---|---|---|---|
| buy_again | 0.072 | 0.759 | 0.303 | 18% |
| global_popularity | 0.041 | 0.703 | 0.192 | 10% |
| dept_popularity | 0.034 | 0.650 | 0.155 | 8% |

Buy-again dominates. Department-localized popularity *underperforms* global —
diversifying away from universal staples costs more base-rate hits than
department affinity recovers. Popularity's respectable hit-rate (0.70) is a
staples effect, not personalization.

**Decisions:** buy-again = primary candidate source + the bar to beat;
global popularity retained as cold-start fallback; department localization
rejected by evidence; personalization must come from behavioral similarity
(CF), not catalog partitions. This notebook's export: `reports/baseline_leaderboard.csv`
(consumed by notebook 20+ as the accumulating scoreboard).